# 04. 선형·다중·다항 회귀
하나의 특성을 사용하는 선형회귀에서 시작해 여러 특성을 사용하는 다중회귀, 다항 특성 생성과 규제까지 연결해서 정리합니다.

> 데이터 분리·기본 평가처럼 이미 익힌 과정은 한 셀에 묶고, 새로 배우는 **특성 확장과 모델 차이**에 집중합니다.

In [ ]:
from common.libraries import *

# 이번 학습에서 집중할 기능
# - LinearRegression : 입력 특성의 선형 결합으로 값을 예측
# - PolynomialFeatures : 제곱항·교차항 등 새로운 특성 생성
# - Ridge / Lasso : 규제를 사용해 모델 복잡도 제어

## 1. 하나의 특성으로 선형회귀

In [ ]:
# 기본 준비: 농어 길이/무게 → 훈련/테스트 분리
perch_length = np.array([8.4, 13.7, 15.0, 16.2, 17.4, 18.0, 18.7, 19.0, 19.6, 20.0, 21.0, 21.0, 21.0, 21.3, 22.0, 22.0, 22.0, 22.0, 22.0, 22.5, 22.5, 22.7, 23.0, 23.5, 24.0, 24.0, 24.6, 25.0, 25.6, 26.5, 27.3, 27.5, 27.5, 27.5, 28.0, 28.7, 30.0, 32.8, 34.5, 35.0, 36.5, 36.0, 37.0, 37.0, 39.0, 39.0, 39.0, 40.0, 40.0, 40.0, 40.0, 42.0, 43.0, 43.0, 43.5, 44.0])
perch_weight = np.array([5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0, 110.0, 115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0, 130.0, 150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0, 197.0, 218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0, 514.0, 556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0, 820.0, 850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0, 1000.0, 1000.0])

X = perch_length.reshape(-1, 1)
y = perch_weight

train_input, test_input, train_target, test_target = train_test_split(
    X, y, test_size=0.25, random_state=42
)

lr = LinearRegression()
lr.fit(train_input, train_target)

print("훈련 R² :", lr.score(train_input, train_target))
print("테스트 R² :", lr.score(test_input, test_target))
print("기울기 :", lr.coef_)
print("절편 :", lr.intercept_)

In [ ]:
# 회귀 직선 확인
plt.scatter(train_input, train_target)
plt.plot(
    [15, 50],
    [15 * lr.coef_[0] + lr.intercept_, 50 * lr.coef_[0] + lr.intercept_]
)
plt.xlabel("length")
plt.ylabel("weight")
plt.show()

## 2. 다항 특성으로 곡선 표현
`LinearRegression` 자체가 자동으로 곡선을 선택하는 것은 아닙니다.  
`x²` 같은 특성을 직접 추가하면 선형회귀가 그 특성들의 선형 결합을 학습하면서 곡선 형태를 표현할 수 있습니다.

In [ ]:
# 길이² 특성을 추가
train_poly = np.column_stack((train_input[:, 0] ** 2, train_input[:, 0]))
test_poly = np.column_stack((test_input[:, 0] ** 2, test_input[:, 0]))

poly_lr = LinearRegression()
poly_lr.fit(train_poly, train_target)

print("훈련 R² :", poly_lr.score(train_poly, train_target))
print("테스트 R² :", poly_lr.score(test_poly, test_target))
print("계수 :", poly_lr.coef_)
print("절편 :", poly_lr.intercept_)

In [ ]:
# 학습한 2차 곡선 확인
point = np.arange(15, 50)
a, b = poly_lr.coef_
c = poly_lr.intercept_

plt.scatter(train_input, train_target)
plt.plot(point, a * point**2 + b * point + c)
plt.xlabel("length")
plt.ylabel("weight")
plt.show()

## 3. 여러 특성을 사용하는 다중회귀
길이뿐 아니라 높이와 너비도 함께 사용합니다.  
여러 입력 특성을 사용한다고 해서 자동으로 '다항회귀'가 되는 것은 아니며, 기본적으로는 여러 특성의 선형 결합을 학습합니다.

In [ ]:
# 기본 준비: 길이·높이·너비 + 무게
perch_full = pd.read_csv("./data/perch_dimensions.csv")
y = np.array([5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0, 110.0, 115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0, 130.0, 150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0, 197.0, 218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0, 514.0, 556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0, 820.0, 850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0, 1000.0, 1000.0])

train_input, test_input, train_target, test_target = train_test_split(
    perch_full, y, test_size=0.25, random_state=42
)

multi_lr = LinearRegression()
multi_lr.fit(train_input, train_target)

print("훈련 R² :", multi_lr.score(train_input, train_target))
print("테스트 R² :", multi_lr.score(test_input, test_target))

## 4. PolynomialFeatures로 특성 확장
`fit()`은 훈련 데이터에서 특성 생성 규칙을 정하고, `transform()`은 같은 규칙을 훈련·테스트 데이터에 적용합니다.  
테스트 데이터에는 다시 `fit()`하지 않습니다.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

poly.fit(train_input)
train_poly = poly.transform(train_input)
test_poly = poly.transform(test_input)

print("원본 특성 :", train_input.shape)
print("확장 특성 :", train_poly.shape)
print(poly.get_feature_names_out())

In [ ]:
poly_lr = LinearRegression()
poly_lr.fit(train_poly, train_target)

train_pred = poly_lr.predict(train_poly)
test_pred = poly_lr.predict(test_poly)

print("훈련 R² :", r2_score(train_target, train_pred))
print("테스트 R² :", r2_score(test_target, test_pred))
print("테스트 MAE :", mean_absolute_error(test_target, test_pred))
print("테스트 MSE :", mean_squared_error(test_target, test_pred))

## 5. 스케일링과 규제
특성이 많아지면 모델이 복잡해질 수 있으므로 StandardScaler로 크기를 맞춘 뒤 Ridge/Lasso로 규제를 적용해 비교합니다.

In [ ]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_poly)
test_scaled = scaler.transform(test_poly)

ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=1.0, max_iter=10000)

for model in [ridge, lasso]:
    model.fit(train_scaled, train_target)
    print(
        model.__class__.__name__,
        "훈련 R² :", round(model.score(train_scaled, train_target), 4),
        "테스트 R² :", round(model.score(test_scaled, test_target), 4),
    )

### 정리
- 선형회귀는 입력 특성들의 **선형 결합**을 학습합니다.
- 곡선 관계를 표현하려면 제곱항 같은 특성을 직접 추가할 수 있습니다.
- `PolynomialFeatures`는 모델이 아니라 **특성 변환기**입니다.
- 특성이 많아질수록 과적합 가능성이 커질 수 있어 Ridge/Lasso 같은 규제를 함께 비교할 수 있습니다.